In [69]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Reading in the data, cleaning and preprocessing(don't worry about this part Mr. Rooney)

In [70]:
# Reading in the data
perf_data = pd.read_csv("../data/NBA_Perf_22.csv", encoding='latin1')
sal_data = pd.read_csv("../data/nba_salaries_22.csv")

# Merging both into one df
merged_data = pd.merge(perf_data, sal_data, on='Player', how='inner')

# Dropping duplicates and N/A values
nba_data = merged_data.drop_duplicates()
nba_data = nba_data.drop('Tm', axis = 1)
nba_data = nba_data.dropna()

# Converting salary from a string to a float
nba_data['Salary_raw'] = nba_data['Salary']
nba_data['Salary'] = nba_data['Salary'].replace(r'[\$,]', '', regex=True)
nba_data['Salary'] = pd.to_numeric(nba_data['Salary'], errors='coerce')

# Finding highest correlation
numeric_vars = nba_data.select_dtypes(include=['float64', 'int64'])
correlation = numeric_vars.corr()
salary_correlation = correlation['Salary'].sort_values(ascending=False)
highest_correlation = salary_correlation[1:10]  # Exclude Salary column itself
highest_correlation_variables = highest_correlation.index.tolist()
highest_correlation_variables


# Scaling the data
scaler = MinMaxScaler()
scaled_numeric_vars = scaler.fit_transform(numeric_vars)
scaled_df = pd.DataFrame(scaled_numeric_vars, columns=numeric_vars.columns, index=numeric_vars.index)
nba_data[numeric_vars.columns] = scaled_df

In [71]:
print(highest_correlation_variables)

['PTS', 'FG', 'FGA', '2PA', 'FT', 'FTA', '2P', 'MP', 'TOV']


Variables such as FG, FGA, 2PA, FT, FTA, and 2P likely have high colinearity concerns. Because of this I will look at PTS and MP against salary.

Although it seems the elbow method indicates two clusters would be a strong choice for k, I want to include more. I feel that having two clusters will not differentiate players well enough. Based on the elbow method graphs and silhouette scores, as well as interpretation in the context of the problem, I am going to proceed with 5 clusters.

In [85]:
kmeans_nba = KMeans(n_clusters=5, random_state=67).fit(x)
nba_data['Cluster'] = kmeans_nba.labels_

## Visualizing to see overpaid and underpaid players:

In [92]:

fig = px.scatter_3d(nba_data, x="PTS", y="MP", z="eFG%", color='Salary',
                    title="NBA Player Clusters based on Performance",
                    hover_name = 'Player',
                    symbol = kmeans_nba.labels_)
fig.update_layout(legend = dict(x=0, y=1, traceorder = 'normal', orientation = 'h'))
fig.update_traces(marker = dict(size = 5))
fig.show()

In [98]:
cluster_centers = pd.DataFrame(kmeans_nba.cluster_centers_, columns=x.columns)
cluster_centers

,Age,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Salary
0,0.273072,0.413487,0.043514,0.287619,0.120317,0.149469,0.461683,0.118013,0.135846,0.303871,...,0.150527,0.133602,0.141026,0.093645,0.189050,0.103355,0.128157,0.241033,0.123516,0.056929
1,0.325413,0.736392,0.645094,0.806526,0.466736,0.564735,0.445980,0.482576,0.504079,0.366841,...,0.159091,0.295561,0.265655,0.342698,0.446281,0.146916,0.371449,0.428803,0.488444,0.299674
2,0.306527,0.718582,0.612883,0.701392,0.417249,0.382185,0.702832,0.096296,0.114398,0.231333,...,0.561315,0.544932,0.581137,0.215337,0.349650,0.347070,0.324786,0.514390,0.386694,0.269522
3,0.320723,0.666283,0.209817,0.559343,0.265387,0.316829,0.477033,0.259627,0.282635,0.344851,...,0.190116,0.247228,0.239326,0.176386,0.335121,0.154392,0.209886,0.354418,0.270732,0.132243
4,0.426290,0.713714,0.716216,0.916911,0.732678,0.820546,0.501513,0.462462,0.500578,0.344568,...,0.221504,0.475878,0.420147,0.578328,0.538084,0.198842,0.625000,0.451737,0.771914,0.703650


In [97]:
cluster_centers[['PTS', 'MP', 'eFG%', 'Salary']]


,PTS,MP,eFG%,Salary
0,0.123516,0.287619,0.586641,0.056929
1,0.488444,0.806526,0.610414,0.299674
2,0.386694,0.701392,0.743351,0.269522
3,0.270732,0.559343,0.622682,0.132243
4,0.771914,0.916911,0.608310,0.703650


- Cluster 0: Benchwarmers — low scoring, low minutes, low salary.(avoid)
- Cluster 1: Solid Starters — respectable scoring and efficiency, high minutes, very cheap. These are undervalued assets(target)
- Cluster 2: Efficient Role Players — not flashy, but highly efficient.(target)
- Cluster 3: Mediocre Contributors — play okay minutes, don’t score much, not very efficient. Could be aging vets or average bench players(avoid)
- Cluster 4: Superstars; expensive but worth the money, probably not able to get them

### Not good choices
- Micheal Porter Jr.
- Kevin Love
- Russell Westbrook

### Good choices
- Ja Morant
- Tyrese Haliburton
- Anthony Edwards

### Backup options
- Saddiq Bey
- Drew Eubanks
- Keldon Johnson

Throughout this process, we grouped players based on their statistics. We then visualized some of the most important statistics(minutes played, points scored, and effective field goal percentage) with the clusters the players are in and their salaries also visible. This graph gives us the ability to see what players are outliers based on how much they are paid for their performance. It is in three dimensions and interactive, so you can see exactly where players lie and choose which statistics you want to visualize. Furthermore, if you wanted you could easily switch the statistics we are visualizing if you wanted to look at something like assists or rebounds.

 I won't get into the details of how we reached this point, but we found that five clusters was the ideal number. This was done by analyzing error and explained variance. From the graph we identified players that are under and over paid, but you are welcome to draw your own conclusions. Clustering allows us to group similar players based on a combination of key metrics, helping uncover undervalued contributors that wouldn’t stand out with raw sorting alone We have also provided descriptions of the individual clusters and what types of players they represent.


### Good Options
 
- Ja Morant: 89th percentile  PTS, 86th percentile eFG%, 62nd percentile MP, 25th percentile salary, cluster 1
- Haliburton: 56th percentile PTS, 73rd percentile eFG%, 95th percentile MP, 9th percentile salary, cluster 1
- Edwards: 70th percentile PTS, 61st percentile eFG%, 90th percentile MP, 22nd percentile salary, cluster 1

These players are clearly not being paid in line with their performance. They are all cluster one players but are in the bottom 25% of the league in salary. It should be noted that these players are likely on their rookie contracts and their pay will increase once they receive a second contract. Given what these players are making right now, they are highly underpaid.

Since these players are on their rookie contracts and may not be attainable given their contract situations, we can shift our focus to players who may not be "superstars" but rather effective role players on cheap contracts. 

### Strong Backup Options

- Saddiq Bey: 51st percentile PTS, 0.86 percentile MP, 0.54 eFG%, 6th percentile salary, cluster 1
- Drew Eubanks: 46th percentile PTS, 76th percentile MP, 85th percentile eFG%, 4th percentile salary, cluster 2
- Keldon Johnson: 54th percentile PTS, 83rd percentile MP, 64th percentile eFG%, 8th percentile salary, cluster 1

These players are all in the lowest 10% of salary across the league, but their stats don't align with this. It should be noted that these players are ranked higher in minutes than points scored, so they may not be the most effective scorers. However, they are above the 50th percentile in eFG%, indicating they are efficient in their scoring, and since they play many minutes they may be good at defending, rebounding, or passing. However, they are cluster 1/2 players paid in the bottom 10% of the league. These are great options for effective role players.

### Bad Options

On the contrary, Michael Porter Jr., Kevin Love, and D'Angelo Russell make a lot of money compared to their performance measured on these metrics. They are paid very high given their performance metrics. 

- MPJ: 30th percentile PTS, 76th percentile MP, 39th percentile eFG%, 64th percentile salary, cluster 3
- Love: 43rd percentile PTS, 56th percentile MP, 66th percentile eFG%, 60th percentile salary, cluster 3
- Westbrook: 59th percentile PTS, 90th percentile MP, 51th percentile eFG%, 97th percentile salary, cluster 4

MPJ plays a ton of minutes but does not take effective field goals or score that much, yet he is a very high paid player. Russell Westbrook is grouped into the superstar category(likely because of his triple doubles) but as one of the highest paid players in the NBA he does not rank nearly high enough for points scored as he should Kevin Love is does not score enough points or play enough minutes to be earning as much money as he is. 

